<a href="https://colab.research.google.com/github/zuhaatawakal-create/AI-/blob/main/Q2_1_(b).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/content/housing.csv')

# Display the first 5 rows and information about the dataset
display(df.head())
display(df.info())

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.6+ MB


None

From the `df.info()` output, we can see that the `total_bedrooms` column has 207 null values (20433 non-null out of 20640 entries). I will fill these missing values with the median of the column, as it's a common approach for numerical features and less sensitive to outliers than the mean.

In [ ]:
# Handle missing values in 'total_bedrooms' by filling with the median
df['total_bedrooms'].fillna(df['total_bedrooms'].median(), inplace=True)

# Verify that there are no more missing values
print('Missing values after imputation:')
print(df.isnull().sum())

Missing values after imputation:
longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
median_house_value    0
ocean_proximity       0
dtype: int64


/tmp/ipykernel_586/668045228.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['total_bedrooms'].fillna(df['total_bedrooms'].median(), inplace=True)


Now that missing values are handled, I will split the data into features (X) and target (y), and then into training and testing sets using an 80:20 split.

In [ ]:
from sklearn.model_selection import train_test_split

# Define features (X) and target (y)
# The 'median_house_value' is typically the target variable in this dataset
X = df.drop('median_house_value', axis=1)
y = df['median_house_value']

# Split the data into training and testing sets (80:20 ratio)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Shape of X_train: {X_train.shape}')
print(f'Shape of X_test: {X_test.shape}')
print(f'Shape of y_train: {y_train.shape}')
print(f'Shape of y_test: {y_test.shape}')

Shape of X_train: (16512, 9)
Shape of X_test: (4128, 9)
Shape of y_train: (16512,)
Shape of y_test: (4128,)


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Identify numerical and categorical features
numerical_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

# Remove the target variable from numerical_features if it accidentally got included
if 'median_house_value' in numerical_features:
    numerical_features.remove('median_house_value')

print(f'Numerical features: {numerical_features}')
print(f'Categorical features: {categorical_features}')

# Create a column transformer for preprocessing
# It will apply StandardScaler to numerical features and OneHotEncoder to categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough'
)

# Apply the preprocessing to the training and testing sets
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f'Shape of processed X_train: {X_train_processed.shape}')
print(f'Shape of processed X_test: {X_test_processed.shape}')

Numerical features: ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income']
Categorical features: ['ocean_proximity']
Shape of processed X_train: (16512, 13)
Shape of processed X_test: (4128, 13)


The features have been successfully preprocessed. The categorical feature `ocean_proximity` has been one-hot encoded, and all numerical features have been scaled. The shape of the processed data now reflects the added columns from one-hot encoding.

In [ ]:
import time
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Initialize the Linear Regression model
linear_model = LinearRegression()

# Measure training time
start_time = time.time()
linear_model.fit(X_train_processed, y_train)
end_time = time.time()
training_time = end_time - start_time

print(f'Training Time: {training_time:.4f} seconds')

# Make predictions on the test set
y_pred = linear_model.predict(X_test_processed)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'Mean Squared Error (MSE): {mse:.2f}')
print(f'R-squared (R²): {r2:.4f}')

Training Time: 0.0255 seconds
Mean Squared Error (MSE): 4908476721.16
R-squared (R²): 0.6254


### Discussion of Scikit-learn Linear Regression

This implementation uses `sklearn.linear_model.LinearRegression`, which provides an optimized and robust way to perform linear regression. Here's a brief discussion of the results and comparison:

*   **Training Time**: Scikit-learn's implementation is highly optimized, often leveraging underlying C++ libraries, leading to very fast training times, especially on larger datasets. The measured training time reflects this efficiency.

*   **Test Accuracy (R²)**: The R-squared value indicates the proportion of the variance in the dependent variable that is predictable from the independent variables. An R² of 1.0 means the model perfectly predicts the target variable. Our current R² value indicates the model's performance on unseen data.

*   **Mean Squared Error (MSE)**: MSE measures the average of the squares of the errors—the average squared difference between the estimated values and the actual value. It's a measure of the quality of an estimator—it is always non-negative, and values closer to zero are better.

**Comparison with other implementations (e.g., a manual implementation)**:

*   **Efficiency**: Scikit-learn's `LinearRegression` is significantly more efficient for large datasets compared to a manual implementation (e.g., using NumPy for matrix operations) due to its optimized algorithms and handling of numerical stability.
*   **Robustness**: It includes features like regularization (though not explicitly used here, it's an option in similar models like `Ridge` or `Lasso`), and handles common issues more gracefully.
*   **Ease of Use**: The API is straightforward, requiring minimal code to train and evaluate a model, reducing the chance of implementation errors compared to writing the algorithm from scratch.
*   **Standardization**: Scikit-learn provides a standardized interface for various machine learning models, making it easy to swap out algorithms and compare them.

In [ ]:
print(f'Training Time: {training_time:.4f} seconds')
print(f'Mean Squared Error (MSE): {mse:.2f}')
print(f'R-squared (R²): {r2:.4f}')

Training Time: 0.0255 seconds
Mean Squared Error (MSE): 4908476721.16
R-squared (R²): 0.6254
